# 02 - Build Deterministic Few-Shot Splits

This notebook creates reproducible training subsets for matched annotation budgets:

- budgets: `50`, `100`, `200`
- seeds: configurable list
- target datasets: `wnut17`, `scierc`

Outputs are written to `data/processed/fewshot_splits/`.

In [1]:
import json
import random
from pathlib import Path
from collections import Counter

import pandas as pd

In [2]:
# Colab helper: mount Google Drive if available
if Path('/content').exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=False)
    except Exception:
        pass

# Path resolution (prefer Google Drive, then Colab local, then local repo)
candidates = [
    Path('/content/drive/MyDrive/AAI590/data/processed'),
    Path('/content/drive/MyDrive/ner_capstone/data/processed'),
    Path('/content/ner_capstone/data/processed'),
    Path.cwd() / 'data' / 'processed',
]

processed_dir = next((p for p in candidates if p.exists()), None)
if processed_dir is None:
    raise FileNotFoundError(
        'Could not find data/processed. Place it in Drive (recommended) or local repo, then rerun.'
    )

fewshot_dir = processed_dir / 'fewshot_splits'
fewshot_dir.mkdir(parents=True, exist_ok=True)

print('Using processed dir:', processed_dir)
print('Few-shot output dir:', fewshot_dir)

Mounted at /content/drive
Using processed dir: /content/drive/MyDrive/AAI590/data/processed
Few-shot output dir: /content/drive/MyDrive/AAI590/data/processed/fewshot_splits


In [3]:
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def save_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=True) + '\n')


def sent_has_entity(sample):
    return any(t != 'O' for t in sample['tags'])


def entity_counter(samples):
    c = Counter()
    for s in samples:
        c.update([t for t in s['tags'] if t != 'O'])
    return c

In [4]:
# Configuration
TARGET_DATASETS = ['wnut17', 'scierc']
SOURCE_SPLIT = 'train'
BUDGETS = [50, 100, 200]
SEEDS = [13, 42, 101]


def stratified_sample_train(samples, budget, seed):
    rng = random.Random(seed)
    positives = [s for s in samples if sent_has_entity(s)]
    negatives = [s for s in samples if not sent_has_entity(s)]

    rng.shuffle(positives)
    rng.shuffle(negatives)

    # Keep some non-entity sentences but prioritize entity coverage
    n_pos = min(len(positives), max(1, int(round(0.8 * budget))))
    n_neg = min(len(negatives), budget - n_pos)

    chosen = positives[:n_pos] + negatives[:n_neg]

    # Fill remaining budget from leftovers if needed
    if len(chosen) < budget:
        used_ids = set(id(x) for x in chosen)
        leftovers = [s for s in samples if id(s) not in used_ids]
        rng.shuffle(leftovers)
        chosen.extend(leftovers[: budget - len(chosen)])

    rng.shuffle(chosen)
    return chosen[:budget]

In [5]:
manifest_rows = []

for ds in TARGET_DATASETS:
    train_fp = processed_dir / ds / f'{ds}_{SOURCE_SPLIT}.jsonl'
    if not train_fp.exists():
        raise FileNotFoundError(f'Missing expected file: {train_fp}')

    samples = load_jsonl(train_fp)
    ds_out_dir = fewshot_dir / ds
    ds_out_dir.mkdir(parents=True, exist_ok=True)

    for budget in BUDGETS:
        if budget > len(samples):
            raise ValueError(f'Budget {budget} > dataset size {len(samples)} for {ds}')

        for seed in SEEDS:
            subset = stratified_sample_train(samples, budget=budget, seed=seed)
            out_fp = ds_out_dir / f'{ds}_train_{budget}_seed_{seed}.jsonl'
            save_jsonl(out_fp, subset)

            tag_counts = entity_counter(subset)
            manifest_rows.append({
                'dataset': ds,
                'source_split': SOURCE_SPLIT,
                'budget': budget,
                'seed': seed,
                'num_sentences': len(subset),
                'num_entity_sentences': sum(sent_has_entity(s) for s in subset),
                'num_entity_tokens': sum(tag_counts.values()),
                'output_file': str(out_fp),
            })

manifest_df = pd.DataFrame(manifest_rows).sort_values(['dataset', 'budget', 'seed'])
manifest_path = fewshot_dir / 'manifest.csv'
manifest_df.to_csv(manifest_path, index=False)

print('Saved manifest:', manifest_path)
display(manifest_df.head(20))

Saved manifest: /content/drive/MyDrive/AAI590/data/processed/fewshot_splits/manifest.csv


,dataset,source_split,budget,seed,num_sentences,num_entity_sentences,num_entity_tokens,output_file
9,scierc,train,50,13,50,40,280,/content/drive/MyDrive/AAI590/data/processed/f...
10,scierc,train,50,42,50,40,292,/content/drive/MyDrive/AAI590/data/processed/f...
11,scierc,train,50,101,50,40,334,/content/drive/MyDrive/AAI590/data/processed/f...
12,scierc,train,100,13,100,80,593,/content/drive/MyDrive/AAI590/data/processed/f...
13,scierc,train,100,42,100,80,615,/content/drive/MyDrive/AAI590/data/processed/f...
14,scierc,train,100,101,100,80,656,/content/drive/MyDrive/AAI590/data/processed/f...
15,scierc,train,200,13,200,160,1228,/content/drive/MyDrive/AAI590/data/processed/f...
16,scierc,train,200,42,200,160,1246,/content/drive/MyDrive/AAI590/data/processed/f...
17,scierc,train,200,101,200,160,1313,/content/drive/MyDrive/AAI590/data/processed/f...
0,wnut17,train,50,13,50,40,105,/content/drive/MyDrive/AAI590/data/processed/f...


In [6]:
# Quick sanity summaries
summary = (
    manifest_df
    .groupby(['dataset', 'budget'], as_index=False)
    .agg(
        avg_entity_sentences=('num_entity_sentences', 'mean'),
        avg_entity_tokens=('num_entity_tokens', 'mean'),
        seeds=('seed', 'count')
    )
)

display(summary)

for ds in TARGET_DATASETS:
    print('\n', ds)
    display(
        manifest_df[manifest_df['dataset'] == ds][['budget', 'seed', 'num_entity_sentences', 'num_entity_tokens']]
        .sort_values(['budget', 'seed'])
    )

,dataset,budget,avg_entity_sentences,avg_entity_tokens,seeds
0,scierc,50,40.0,302.000000,3
1,scierc,100,80.0,621.333333,3
2,scierc,200,160.0,1262.333333,3
3,wnut17,50,40.0,100.666667,3
4,wnut17,100,80.0,202.333333,3
5,wnut17,200,160.0,414.333333,3



 wnut17


,budget,seed,num_entity_sentences,num_entity_tokens
0,50,13,40,105
1,50,42,40,79
2,50,101,40,118
3,100,13,80,198
4,100,42,80,182
5,100,101,80,227
6,200,13,160,403
7,200,42,160,408
8,200,101,160,432



 scierc


,budget,seed,num_entity_sentences,num_entity_tokens
9,50,13,40,280
10,50,42,40,292
11,50,101,40,334
12,100,13,80,593
13,100,42,80,615
14,100,101,80,656
15,200,13,160,1228
16,200,42,160,1246
17,200,101,160,1313


## Notes for Experiment Consistency

- Use the same `budget` and `seed` combinations for all adaptation strategies.
- Keep validation/test unchanged; only subset training data.
- Report average and standard deviation over seeds for each method-budget pair.